# TOX3GNN – Optuna + 30-Run  Experiment
 **T4 GPU**

## 1 · Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Edit this to match where your TOX folder lives in Drive ───
# DRIVE_ROOT = '/content/drive/MyDrive/AUA/Thesis/code/HybridGNN/TOX'
DRIVE_ROOT = '/content/drive/MyDrive/Research/HybridGNN/TOX'

# tanh()
# os.makedirs(f'{DRIVE_ROOT}/checkpoints_tox21', exist_ok=True)
# os.makedirs(f'{DRIVE_ROOT}/results_tox21', exist_ok=True)

os.makedirs(f'{DRIVE_ROOT}/checkpoints_tox21_leakyrelu', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results_tox21_leakyrelu', exist_ok=True)

print('Drive mounted. Root:', DRIVE_ROOT)
print('Contents:', os.listdir(DRIVE_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Root: /content/drive/MyDrive/Research/HybridGNN/TOX
Contents: ['tox21_dataset.csv', 'results_tox21', 'checkpoints_tox21', '__pycache__', 'SR-ARE', 'Analysis', 'tox21_analysis.ipynb', 'checkpoints_tox21_leakyrelu', 'results_tox21_leakyrelu', 'utils.py', 'TOX3GNN_multitask.ipynb', 'TOX3GNN_multitask_leaky.ipynb']


## 2 · Install dependencies

In [2]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'optuna'],  check=True)

import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}')
print('Dependencies ready.')

torch=2.11.0+cu128  cuda=True
Dependencies ready.


## 3 · Imports

In [3]:
import sys, time, random, shutil, warnings, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import optuna

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, GATConv, GINConv, SAGEConv
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
from torch_geometric.loader import DataLoader

from rdkit.Chem.rdmolops import GetAdjacencyMatrix
import torch
from torch_geometric.data import Data

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from rdkit import Chem

import csv, glob, json
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


In [4]:
# utils.py must be in DRIVE_ROOT alongside tox21_dataset.csv
sys.path.insert(0, DRIVE_ROOT)

from utils import (
    one_hot_encoding,
    get_atom_features,
    get_bond_features,
    smiles_to_graph_list,
    scaffold_split,
    get_valid_mask,
    save_ckp,
    load_ckp,
    optimizer_to,
    round_to_4,
    log_weights_to_tensorboard,
    inspect_model_weights
)

# Alias used throughout this notebook



print('utils.py loaded from Drive — using your exact implementations.')


utils.py loaded from Drive — using your exact implementations.


## 5 · Configuration

In [5]:
#  Paths
DATA_CSV  = f'{DRIVE_ROOT}/tox21_dataset.csv'

# CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints_tox21'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints_tox21_leakyrelu'

TASK_NAME = 'SR-ARE'

# RESULTS_DIR = f'{DRIVE_ROOT}/{TASK_NAME}/results_tox21'

# RESULTS_DIR  = f'{DRIVE_ROOT}/{TASK_NAME}/results_tox21_optuna'
RESULTS_DIR  = f'{DRIVE_ROOT}/{TASK_NAME}/results_tox21_leakyrelu'

# ── Optuna toggle ─────────────
# Set RUN_OPTUNA = True to search for hyperparameters first.
RUN_OPTUNA = False
N_TRIALS     = 20      # number of Optuna trials (each trains 50 epochs)

# Change to False if you are running on the full dataset
IS_DROPPING = True
# ── Known-best params (used when RUN_OPTUNA = False, or as Optuna fallback) ──
KNOWN_LAYER_TYPES = ['sage', 'gin', 'gin']
KNOWN_HIDDEN = 520
KNOWN_DROPOUT = np.round(0.43479245061434324, 4)
KNOWN_LR = np.round(0.0003585966617440445, 5)
KNOWN_WD = 1e-4

# ── 30-run experiment settings ──────────────
N_RUNS = 10
MAX_EPOCHS = 500
EVAL_EVERY = 10
PATIENCE = 20
NUM_GRAPHS_PER_BATCH = 64
USE_SCAFFOLD_SPLIT = True
GLOBAL_SEED = 43

# set it to your chosen task

print('Config OK')
print(f'  RUN_OPTUNA={RUN_OPTUNA}  N_TRIALS={N_TRIALS}')
print(f'  scaffold_split={USE_SCAFFOLD_SPLIT}  N_RUNS={N_RUNS}  MAX_EPOCHS={MAX_EPOCHS}')


Config OK
  RUN_OPTUNA=False  N_TRIALS=20
  scaffold_split=True  N_RUNS=10  MAX_EPOCHS=500


## 6 · Load dataset & build splits

In [6]:
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

# Read the imputed tox21 or the original tox21 data
df_raw= pd.read_csv(DATA_CSV)
print(f'Size of the Dataset: {len(df_raw)}\n')

# All task columns — everything except smiles/mol_id
task_cols = [col for col in df_raw.columns if col not in ['smiles', 'mol_id']]
NUM_TASKS = len(task_cols)
print(f'Tasks ({NUM_TASKS}): {task_cols}')

# Dtype casting
df_raw[['smiles', 'mol_id']] = df_raw[['smiles', 'mol_id']].astype('str')
df_raw[task_cols] = df_raw[task_cols].apply(pd.to_numeric, errors = 'coerce').astype('float32')


def prep_data(df_source:pd.DataFrame, IS_DROPPING:bool, target_task:str = TASK_NAME):
  """
  Data preperation: Drops the null values (IS_DROPPING = True)
  and uses validation mask to filter out invalid SMILES
  Parameters:
  df_source: pd.DataFrame
  IS_DROPPING: bool
  target_task: str
  """
  print(f'Prepping the Tox21 Task {TASK_NAME} Data...')
  df_task = df_source[['smiles', target_task]].copy()
  if IS_DROPPING:
    print('Dropping the null values...')
    df_task = df_task.dropna(subset=[target_task]).reset_index(drop=True)
    print(f'Dropped {len(df_task)} rows')
  else:
    print('Continuing without Dropping the NUlls')

  print("\nCreating the SMILES validation mask...")
  smiles_valid_mask = get_valid_mask(df_task['smiles'])

  df_clean = df_task[smiles_valid_mask].reset_index(drop=True)
  print(f'Valid SMILES: {len(df_clean)} / {len(df_task)}')

  X_smiles = df_clean['smiles'].tolist()
  y_labels = df_clean[[target_task]].values.astype(np.float32)

# Multi-task extraction of X, y
  # X_smiles, y_labels = [], []
  # for _, row in df_clean.iterrows():
  #   smi = row['smiles']
  #   X_smiles.append(smi)
  #   y_labels.append(row[target_task])
  # y_labels = np.array(y_labels)
  print(f'Label matrix shape: {y_labels.shape}')

  print('Building molecular graphs...')
  data_list = smiles_to_graph_list(X_smiles, y_labels)
  print(f'Graph list: {len(data_list)} molecules  |  y shape per graph: {data_list[0].y.shape}')
  # assert len(data_list) == len(X_smiles)

  return data_list, X_smiles, y_labels

# data_list, X_smiles, y_labels = prep_data(df_raw,IS_DROPPING, TASK_NAME)


def split_data(data_list, X_smiles, train_frac:float, val_frac:float,
               y_labels, USE_SCAFFOLD_SPLIT:bool=True):
  """
   Splits the data: scaffold split (USE_SCAFFOLD_SPLIT=True) or a random split(USE_SCAFFOLD_SPLIT=False)
  Parameters:

  data_list: pd.DataFrame
  X_smiles
  """
  if USE_SCAFFOLD_SPLIT:
      train_idx, val_idx, test_idx = scaffold_split(X_smiles, train_frac=train_frac,
                                                    val_frac=val_frac, seed=GLOBAL_SEED)
  else:
      all_idx = list(range(len(data_list)))
      test_size = 1.0 - train_frac
      train_idx, temp = train_test_split(all_idx, test_size=test_size, random_state=GLOBAL_SEED)
      val_idx, test_idx = train_test_split(temp, test_size=0.5, random_state=GLOBAL_SEED)
      print(f'Random split → train:{len(train_idx)} val:{len(val_idx)} test:{len(test_idx)}')

  val_loader  = DataLoader([data_list[i] for i in val_idx],
                          batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False, drop_last=False)
  test_loader = DataLoader([data_list[i] for i in test_idx],
                          batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False, drop_last=False)
  trainval_data = [data_list[i] for i in list(train_idx) + list(val_idx)]
  train_data    = [data_list[i] for i in train_idx]

  print('Data splits ready.')
  return train_data, val_loader, test_loader, trainval_data, train_idx, val_idx, test_idx

# Version 2 to avoid probable data leakage
def prep_single_split(df_source: pd.DataFrame, IS_DROPPING: bool, target_task: str = TASK_NAME):
    df_task = df_source[['smiles', target_task]].copy()

    if IS_DROPPING:
        df_task = df_task.dropna(subset=[target_task]).reset_index(drop=True)

    smiles_valid_mask = get_valid_mask(df_task['smiles'])
    df_clean = df_task[smiles_valid_mask].reset_index(drop=True)

    X_smiles = df_clean['smiles'].tolist()
    y_labels = df_clean[[target_task]].values.astype(np.float32)

    data_list = smiles_to_graph_list(X_smiles, y_labels)
    return data_list, X_smiles, y_labels


def split_raw_data(df_raw: pd.DataFrame, train_frac: float, val_frac: float, USE_SCAFFOLD_SPLIT: bool = True):
    smiles_list = df_raw['smiles'].tolist()

    if USE_SCAFFOLD_SPLIT:
        train_idx, val_idx, test_idx = scaffold_split(
            smiles_list, train_frac=train_frac, val_frac=val_frac, seed=GLOBAL_SEED
        )
    else:
        all_idx = list(range(len(df_raw)))
        test_size = 1.0 - train_frac
        train_idx, temp_idx = train_test_split(all_idx, test_size=test_size, random_state=GLOBAL_SEED)

        # Calculate relative split for remaining validation and test sets
        val_relative_ratio = val_frac / test_size
        val_idx, test_idx = train_test_split(temp_idx, train_size=val_relative_ratio, random_state=GLOBAL_SEED)

    df_train = df_raw.iloc[train_idx].reset_index(drop=True)
    df_val   = df_raw.iloc[val_idx].reset_index(drop=True)
    df_test  = df_raw.iloc[test_idx].reset_index(drop=True)
    df_trainval = pd.concat([df_train, df_val], ignore_index=True)

    return df_train, df_val, df_test, df_trainval

Size of the Dataset: 7831

Tasks (12): ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


In [7]:
TRAIN_FRAC = 0.8
VAL_FRAC = (1.0 - TRAIN_FRAC) / 2.0
# Scaffold or Random splits
df_train, df_val, df_test, df_trainval = split_raw_data(
    df_raw, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, USE_SCAFFOLD_SPLIT=USE_SCAFFOLD_SPLIT
)

train_data, X_train, y_train = prep_single_split(df_train, IS_DROPPING, TASK_NAME)
val_data,   X_val,   y_val   = prep_single_split(df_val,   IS_DROPPING, TASK_NAME)
test_data,  X_test,  y_test  = prep_single_split(df_test,  IS_DROPPING, TASK_NAME)
trainval_data, _, _          = prep_single_split(df_trainval, IS_DROPPING, TASK_NAME)
# DataLoaders
train_loader = DataLoader(train_data, batch_size=NUM_GRAPHS_PER_BATCH, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=NUM_GRAPHS_PER_BATCH, shuffle=False)

total_final = len(train_data) + len(val_data) + len(test_data)

print("\n" + "="*55)
print(f"FINAL PROCESSED DATASET SUMMARY ({TASK_NAME})")
print("="*55)
print(f"Train set:     {len(train_data):>6d} graphs ({len(train_data)/total_final:.2%})")
print(f"Val set:       {len(val_data):>6d} graphs ({len(val_data)/total_final:.2%})")
print(f"Test set:      {len(test_data):>6d} graphs ({len(test_data)/total_final:.2%})")
print(f"Train+Val set: {len(trainval_data):>6d} graphs")
print("-" * 55)
print(f"Total Model-Ready Samples: {total_final} / {len(df_raw)} raw samples")
print("="*55 + "\n")

[15:29:04] WARNING: not removing hydrogen atom without neighbors
[15:29:04] Explicit valence for atom # 8 Al, 6, is greater than permitted


Cant convert to Mol: NC(=O)NC1N=C(O[AlH3](O)O)NC1=O



[15:29:05] Explicit valence for atom # 3 Al, 6, is greater than permitted
[15:29:05] Explicit valence for atom # 4 Al, 6, is greater than permitted


Cant convert to Mol: O=CO[AlH3](OC=O)OC=O

Cant convert to Mol: CC(=O)O[AlH3](O)O



[15:29:06] Explicit valence for atom # 4 Al, 6, is greater than permitted


...

Cant convert to Mol: CCCCCCCCCCCCCCCCCC(=O)O[AlH3](O)O

Scaffold split → train: 6264 (79.99%), val: 783 (10.00%), test: 784 (10.01%)


[15:29:09] Explicit valence for atom # 8 Al, 6, is greater than permitted
[15:29:09] Explicit valence for atom # 3 Al, 6, is greater than permitted
[15:29:09] Explicit valence for atom # 4 Al, 6, is greater than permitted
[15:29:09] Explicit valence for atom # 4 Al, 6, is greater than permitted
[15:29:09] Explicit valence for atom # 9 Al, 6, is greater than permitted
[15:29:09] Explicit valence for atom # 5 Al, 6, is greater than permitted
[15:29:09] Explicit valence for atom # 16 Al, 6, is greater than permitted


Total # SMILES/Valid # SMILES: 1.0
Total # SMILES/Valid # SMILES: 1.0
Total # SMILES/Valid # SMILES: 1.0


[15:29:30] Explicit valence for atom # 8 Al, 6, is greater than permitted
[15:29:30] Explicit valence for atom # 3 Al, 6, is greater than permitted
[15:29:30] Explicit valence for atom # 4 Al, 6, is greater than permitted
[15:29:30] Explicit valence for atom # 4 Al, 6, is greater than permitted
[15:29:30] Explicit valence for atom # 9 Al, 6, is greater than permitted
[15:29:30] Explicit valence for atom # 5 Al, 6, is greater than permitted
[15:29:30] Explicit valence for atom # 16 Al, 6, is greater than permitted


Total # SMILES/Valid # SMILES: 1.0

FINAL PROCESSED DATASET SUMMARY (SR-ARE)
Train set:       4695 graphs (80.60%)
Val set:          561 graphs (9.63%)
Test set:         569 graphs (9.77%)
Train+Val set:   5256 graphs
-------------------------------------------------------
Total Model-Ready Samples: 5825 / 7831 raw samples



## 7 · HybridGNN model

In [24]:
class HybridGNN(torch.nn.Module):
    def __init__(self, layer_types, hidden_dim, dropout, n_tasks=1):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        in_dim = 79
        for lt in layer_types:
            if lt == 'gcn':
                self.convs.append(GCNConv(in_dim, hidden_dim))
            elif lt == 'gat':
                self.convs.append(GATConv(in_dim, hidden_dim))
            elif lt == 'gin':
                mlp = nn.Sequential(
                    nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
                    nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
                self.convs.append(GINConv(mlp, eps=0.00005, train_eps=True))
            elif lt == 'sage':
                self.convs.append(SAGEConv(in_dim, hidden_dim))
            else:
                raise ValueError(f'Unknown layer type: {lt}')
            in_dim = hidden_dim
        self.drop = nn.Dropout(p=dropout)
        # n_tasks outputs — one logit per task
        self.out  = Linear(hidden_dim * 2, n_tasks)

    def forward(self, x, edge_index, batch_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.leaky_relu(x)
        x = self.drop(x)
        x = torch.cat([gmp(x, batch_index), gap(x, batch_index)], dim=1)
        return self.out(x)


def multitask_loss(logits, targets, pos_weight_tensor):
    """
    BCE loss over all tasks, masking out NaN labels.
    logits  : (batch, n_tasks)
    targets : (batch, n_tasks)  — may contain NaN
    pos_weight_tensor: (n_tasks,)
    """
    valid_mask = ~torch.isnan(targets)          # (batch, n_tasks) bool
    if valid_mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True, device=logits.device)
    # Replace NaN with 0 so BCE doesn't error — masked out anyway
    targets_clean = targets.clone()
    targets_clean[~valid_mask] = 0.0
    # Compute per-element BCE with per-task pos_weight
    criterion = torch.nn.BCEWithLogitsLoss(
        # -[w*y*sig(y)+w*(1-y)*sig(1-y)]
        pos_weight=pos_weight_tensor, reduction='none')
    loss_all = criterion(logits, targets_clean)  # (batch, n_tasks)
    # Zero out the NaN positions and average over valid only
    loss_all = loss_all * valid_mask.float()
    return loss_all.sum() / valid_mask.float().sum()


def multi_evaluate_auc(model, loader, device, task_cols):
    """
    Mean ROC-AUC across all tasks, skipping tasks where the loader
    has fewer than 2 unique labels (can't compute AUC).
    """
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x.float(), batch.edge_index, batch.batch)
            all_logits.append(logits.cpu().numpy())
            all_targets.append(batch.y.cpu().numpy())
    logits_np  = np.concatenate(all_logits,  axis=0)   # (N, n_tasks)
    targets_np = np.concatenate(all_targets, axis=0)   # (N, n_tasks)


    task_aucs = []
    for t_idx, t_name in enumerate(task_cols):
        y_true = targets_np[:, t_idx]
        y_score = torch.sigmoid(torch.tensor(logits_np[:, t_idx])).numpy()
        valid   = ~np.isnan(y_true)
        if valid.sum() < 2 or len(np.unique(y_true[valid])) < 2:
            continue   # skip tasks with no positive or no negative examples
        task_aucs.append(roc_auc_score(y_true[valid], y_score[valid]))

    return float(np.mean(task_aucs)) if task_aucs else float('nan')

def evaluate_auc(model, loader, device, task_cols):
    """
    Mean ROC-AUC across all tasks, skipping tasks where the loader
    has fewer than 2 unique labels (can't compute AUC).
    """
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x.float(), batch.edge_index, batch.batch)
            all_logits.append(logits.view(-1).cpu().numpy())
            all_targets.append(batch.y.view(-1).cpu().numpy())
    logits_np  = np.concatenate(all_logits,  axis=0)
    targets_np = np.concatenate(all_targets, axis=0)

    y_pred =torch.sigmoid(torch.tensor(logits_np)).numpy()
    y_true = targets_np

    valid = ~np.isnan(y_true)
    y_true_valid = y_true[valid]
    y_pred_valid = y_pred[valid]
    y_unique = np.unique(y_true_valid)
    if len(y_unique) < 2:
        print(f"Unique values in Target Y: {y_unique}")
        return float('nan')

    return float(roc_auc_score(y_true_valid, y_pred_valid))

print('HybridGNN and helpers defined.')
print(f'Output layer: {len(task_cols) if "task_cols" in dir() else "n_tasks"} tasks')

HybridGNN and helpers defined.
Output layer: 12 tasks


## 8 · Optuna hyperparameter search

In [25]:
# Positional Argument
# TODO devide to DEVICE
def train_target_pw_data_list(data_list, train_idx, device = DEVICE):
  train_targets = np.array([data_list[i].y.item() for i in train_idx])
  pos_count = np.sum(train_targets == 1)
  neg_count = np.sum(train_targets == 0)
  pos_weight = torch.tensor([neg_count / max(pos_count, 1)], dtype=torch.float32).to(device)
  return train_targets, pos_weight

def train_target_pw_graph(train_data, device=DEVICE):
    """
    Creates a positional weight to compensate
    for the class imbalance in the loss function.

    Parameters:
    train_data: list of PyG Data objects
    device: torch.device

    Returns:
    train_targets: torch.tensor
    pos_weight: torch.tensor
    """
    targets_list = [
        graph.y.view(-1) for graph in train_data
        if graph.y is not None and not torch.isnan(graph.y).all()
    ]

    if not targets_list:
        raise ValueError("train_data contains no valid target values (y).")

    # concatenate into 1D PyTorch Tensor
    train_targets = torch.cat(targets_list, dim=0).float()
    pos_count = (train_targets == 1).sum().item()
    neg_count = (train_targets == 0).sum().item()

    pos_weight_val = neg_count / max(pos_count, 1)
    pos_weight = torch.tensor([pos_weight_val], dtype=torch.float32, device=device)

    return train_targets, pos_weight
def multi_task_pos_weight(data_list, train_idx, num_tasks = NUM_TASKS, device = DEVICE):
  train_targets = np.array([data_list[i].y.detach().cpu().reshape(-1).numpy() for i in train_idx])
  print(f"train_targets shape: {train_targets.shape}")
  pw_list = []
  for col in range(num_tasks):
      col_vals  = train_targets[:, col]
      valid      = ~np.isnan(col_vals)
      pos_count = (col_vals[valid] == 1).sum()
      neg_count = (col_vals[valid] == 0).sum()
      weight    = neg_count / max(pos_count, 1) if pos_count > 0 else 1.0
      pw_list.append(weight)
      print(f'  {task_cols[col]:<15}: pos={pos_count}  neg={neg_count}  weight={weight:.2f}')

  pos_weight_tensor = torch.tensor(pw_list, dtype=torch.float32).to(device)
  print(f'\npos_weight_tensor shape: {pos_weight_tensor.shape}  (one per task)')
  return train_targets, pos_weight_tensor

In [26]:
# ── Optuna (single-task proxy on for speed; full multi-task too slow) ──
def objective(trial):
    optuna_train_loader = DataLoader(
        train_data,batch_size=NUM_GRAPHS_PER_BATCH,
        shuffle=True, drop_last=True)

    layer_types = [trial.suggest_categorical(f'layer_{i}', ['gcn','gat','gin','sage'])
                    for i in range(3)]
    hidden_dim  = trial.suggest_int('hidden_dim', 200, 532, step=32)
    dropout     = trial.suggest_float('dropout', 0.0, 0.5)
    lr     = trial.suggest_float('lr', 1e-5, 1e-3, log=True)

    model_optuna = HybridGNN(layer_types, hidden_dim, dropout, n_tasks=1).to(DEVICE)
    opt  = torch.optim.Adam(model_optuna.parameters(), lr=lr)
    best_val, patience_ctr = 0.0, 0

    for epoch in range(100):
        model_optuna.train()
        for batch in optuna_train_loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            logits = model_optuna(batch.x.float(), batch.edge_index, batch.batch)
            # loss   = multitask_loss(logits, batch.y, pos_weight_tensor)
            loss = F.binary_cross_entropy_with_logits(
                logits.view(-1), batch.y.view(-1), pos_weight=pos_weight_tensor)

            loss.backward()
            stats = inspect_model_weights(model)
            print(f"Conv1 grad norm: {stats['convs.0.weight']['grad_norm']:.6f}")
            opt.step()

        val_auc = evaluate_auc(model_optuna, val_loader, DEVICE, task_cols)

        trial.report(val_auc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        if val_auc > best_val:
            best_val, patience_ctr = val_auc, 0
        else:
            patience_ctr += 1
            if patience_ctr >= 5:
                break
    return best_val

def optuna_get_best_params(n_trials: int = N_TRIALS, run_optuina:bool=RUN_OPTUNA):
  if run_optuina:

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    study  = optuna.create_study(direction='maximize', pruner=pruner)
    print(f'Running Optuna ({N_TRIALS} trials)...')
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best_p= study.best_trial.params
    best_layer_types = [best_p[f'layer_{i}'] for i in range(3)]
    best_hidden = best_p['hidden_dim']
    best_dropout = np.round(best_p['dropout'], 4)
    best_lr = np.round(best_p['lr'], 5)
    print(f'Best AUC: {study.best_trial.value:.4f}  layers={best_layer_types}')

    from datetime import datetime
    optuna_dir = os.path.join(RESULTS_DIR, 'optuna')
    os.makedirs(optuna_dir, exist_ok=True)
    optuna_path = os.path.join(optuna_dir, f'optuna_results_{datetime.today().strftime("%Y-%m-%d")}.txt')

    with open(optuna_path, 'w') as f:
        f.write('Optuna Study Results\n' + '='*60 + '\n')
        f.write(f'Best AUC: {study.best_trial.value:.6f}\n')
        for k, v in study.best_trial.params.items():
            f.write(f'  {k}: {v}\n')
        f.write('\nAll trials:\n')
        for rank, t in enumerate(sorted(
            [t for t in study.trials if t.value is not None],
            key=lambda t: t.value, reverse=True), 1):
            f.write(f'{rank}. Trial #{t.number}: AUC={t.value:.6f} | {t.params}\n')
    print(f'Optuna results saved to {optuna_path}')

  else:
      best_layer_types = KNOWN_LAYER_TYPES
      best_hidden = KNOWN_HIDDEN
      best_dropout = KNOWN_DROPOUT
      best_lr = KNOWN_LR
      print(f'Using known-best params: {best_layer_types}  hidden={best_hidden}')
  return best_layer_types, best_hidden, best_dropout, best_lr


In [27]:
best_layer_types, best_hidden, best_dropout, best_lr = optuna_get_best_params(n_trials = N_TRIALS, run_optuina=RUN_OPTUNA)

Using known-best params: ['sage', 'gin', 'gin']  hidden=520


In [28]:
print(f'Best Layer Types: {best_layer_types}')
print(f'Best Hidden: {best_hidden}')
print(f'Best Dropout: {best_dropout}')
print(f'Best LR: {best_lr}')

Best Layer Types: ['sage', 'gin', 'gin']
Best Hidden: 520
Best Dropout: 0.4348
Best LR: 0.00036


## 9 · Save helpers

In [29]:
def append_task_run_result(results_dir, task_name:str,
run_id, test_auc, best_val_auc, auc_history, train_loss_history):

    """Safely appends structured JSON arrays to a task-specific CSV without parser errors."""
    os.makedirs(results_dir, exist_ok=True)
    # Generates filenames like 'SR-ARE_runs_summary.csv'
    csv_filename = f"{task_name}_runs_summary.csv"
    csv_path = os.path.join(results_dir, csv_filename)
    file_exists = os.path.exists(csv_path)

    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['task', 'model', 'run', 'test_auc', 'best_val_auc', 'train_loss_history', 'val_auc_history'])
        writer.writerow([
            task_name, MOD_NAME, run_id, f'{test_auc:.6f}', f'{best_val_auc:.6f}',
            json.dumps(train_loss_history), json.dumps(auc_history)
        ])

## 10 · Training


In [30]:
# Set the Task Checkpoint
# MOD_NAME = '_'.join(best_layer_types)
MOD_NAME = '_'.join(best_layer_types + ['leakyrelu'])
def make_task_ckp(ckp_dir:str = CHECKPOINT_DIR, current_task:str=TASK_NAME, model_name:str= MOD_NAME):
  """
  Creates the model and model_best dir inside the
  TASK_NAME dir.
  Parameters:
  CHECKPOINT_DIR: str
  current_task: str
  Returns:
  task_ckpt_dir: str
  task_model_dir: str

  Returns:
  task_ckpt_dir: str
  task_model_dir: str
  """
  task_ckpt_dir  = os.path.join(ckp_dir, current_task, model_name)
  task_model_dir = os.path.join(ckp_dir, current_task, model_name + '_best')
  os.makedirs(task_ckpt_dir, exist_ok=True)
  os.makedirs(task_model_dir, exist_ok=True)
  return task_ckpt_dir, task_model_dir


In [31]:
# def train_target_pw_graph(train_data, device=DEVICE):
#   targets_list = [graph.y.view(-1) for graph in train_data
#         if graph.y is not None and not torch.isnan(graph.y).all()]
#   # Fixed: Use torch.sum instead of np.sum for torch.Tensor
#   pos_count = torch.sum(train_targets == 1)
#   neg_count = torch.sum(train_targets == 0)
#   # Fixed: Convert pos_count to a Python scalar using .item() for the max function
#   pos_weight = torch.tensor([neg_count.item() / max(pos_count.item(), 1)], dtype=torch.float32).to(device)
#   return train_targets, pos_weight

In [32]:
from torch.utils.tensorboard import SummaryWriter


In [39]:
# mean-across-tasks test AUC per run
all_run_aucs = []
train_targets, pos_weight_tensor = train_target_pw_graph(train_data, device=DEVICE)

experiment_start = time.time()
task_ckpt_dir, task_model_dir = make_task_ckp(CHECKPOINT_DIR, TASK_NAME, MOD_NAME)

def train_test_model(
    model_cls,
    model_kwargs: dict,
    optim_obj,
    lr: float,
    device: str = DEVICE,
    n_runs: int = N_RUNS,
    task_cols: list = task_cols,
):

    for run in range(n_runs):
        run_start = time.time()

        log_dir = os.path.join(RESULTS_DIR, TASK_NAME, MOD_NAME, f'run_{run:02d}')
        writer = SummaryWriter(log_dir=log_dir)
        run_seed = run
        torch.manual_seed(run_seed)
        np.random.seed(run_seed)
        random.seed(run_seed)

        train_loader = DataLoader(
            train_data,
            batch_size=NUM_GRAPHS_PER_BATCH,
            shuffle=True,
            drop_last=True,
            generator=torch.Generator().manual_seed(run_seed),
        )

        best_val_auc = 0.0
        patience_ctr = 0
        auc_history = []
        train_loss_list = []

        # Instantiate model and optimizer for the current run using the passed arguments
        model = model_cls(**model_kwargs).to(device)
        optimizer = optim_obj(model.parameters(), lr=lr, weight_decay=KNOWN_WD)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=6, min_lr=1e-6
        )
        for epoch in range(MAX_EPOCHS):
            model.train()
            epoch_loss = 0.0
            num_batches = 0
            for batch in train_loader:
                batch = batch.to(DEVICE)
                optimizer.zero_grad()
                logits = model(batch.x.float(), batch.edge_index, batch.batch)
                loss = F.binary_cross_entropy_with_logits(
                    logits.view(-1), batch.y.view(-1), pos_weight=pos_weight_tensor
                )
                loss.backward()

                stats = inspect_model_weights(model)
                optimizer.step()

                epoch_loss += loss.item()
                num_batches += 1
            epoch_mean_loss = epoch_loss / num_batches
            train_loss_list.append(round(epoch_mean_loss, 6))

            writer.add_scalar('Loss/train', epoch_mean_loss, epoch + 1)
            writer.add_scalar(
                'Learning Rate', optimizer.param_groups[0]['lr'], epoch + 1
            )

            if (epoch + 1) % EVAL_EVERY == 0:
                log_weights_to_tensorboard(writer, model, step=epoch + 1)
                val_auc = evaluate_auc(model, val_loader, DEVICE, task_cols)
                auc_history.append(round(val_auc, 6))

                writer.add_scalar('AUC/val', val_auc, epoch + 1)
                scheduler.step(val_auc)

                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    patience_ctr = 0
                    state = {
                        'epoch': epoch + 1,
                        'state_dict': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'val_auc': best_val_auc,
                        'epoch_mean_loss': epoch_mean_loss,
                        'train_history': train_loss_list,
                        'auc_history': auc_history,
                        'layers': best_layer_types,
                        'hidden': best_hidden,
                        'dropout': best_dropout,
                        'lr': best_lr,
                    }
                    save_ckp(
                        state,
                        True,
                        task_ckpt_dir,
                        task_model_dir,
                        f'model_run{run:02d}.pt',
                        f'best_model_run{run:02d}.pt',
                    )
                    print(
                        f'  [DEBUG] After save_ckp for run {run:02d}, best_model_dir'
                        f' contents: {os.listdir(task_model_dir)}'
                    )
                else:
                    patience_ctr += 1

                if patience_ctr >= PATIENCE:
                    print(
                        f'  Run {run:02d} | Early stop @ ep {epoch+1} | best Val AUC'
                        f' {best_val_auc:.4f}'
                    )
                    break

        # FIXED INDENT: 8 spaces relative to def (inside 'for run')
        best_ckpt = os.path.join(task_model_dir, f'best_model_run{run:02d}.pt')
        if os.path.exists(best_ckpt):
            model, optimizer, _ = load_ckp(best_ckpt, model, optimizer)

        # Evaluate Test AUC
        test_auc = evaluate_auc(model, test_loader, DEVICE, task_cols)
        auc_history.append(round(test_auc, 6))
        all_run_aucs.append(test_auc)

        writer.add_scalar('AUC/test', test_auc, run)
        writer.close()
        run_time_sec = time.time() - run_start

        run_mins = run_time_sec / 60
        total_mins = (time.time() - experiment_start) / 60
        print(
            f'Run {run+1:02d} | Task: {TASK_NAME:<12} | Test AUC: {test_auc:.4f} |'
            f' Val AUC: {best_val_auc:.4f} | Run Time: {run_mins:.2f}m | Total Time:'
            f' {total_mins:.2f}m'
        )

    return test_auc, best_val_auc, auc_history, train_loss_list

In [40]:
print(f'Model : {MOD_NAME}  |  Tasks: {TASK_NAME}  |  Runs: {N_RUNS}')
model_init_kwargs = {'layer_types': best_layer_types, 'hidden_dim': best_hidden, 'dropout': best_dropout, 'n_tasks': 1}
test_auc, best_val_auc, auc_history, train_loss_list = train_test_model(
    model_cls=HybridGNN,
    model_kwargs=model_init_kwargs,
    optim_obj=torch.optim.AdamW,
    lr=best_lr,
    device=DEVICE,
    n_runs=N_RUNS,
    task_cols=task_cols
)

Model : sage_gin_gin_leakyrelu  |  Tasks: SR-ARE  |  Runs: 10
  [DEBUG] After save_ckp for run 00, best_model_dir contents: ['best_model_run00.pt']
  [DEBUG] After save_ckp for run 00, best_model_dir contents: ['best_model_run00.pt']
  [DEBUG] After save_ckp for run 00, best_model_dir contents: ['best_model_run00.pt']
  [DEBUG] After save_ckp for run 00, best_model_dir contents: ['best_model_run00.pt']
  Run 00 | Early stop @ ep 280 | best Val AUC 0.7662
Run 01 | Task: SR-ARE       | Test AUC: 0.7573 | Val AUC: 0.7662 | Run Time: 4.06m | Total Time: 4.09m
  [DEBUG] After save_ckp for run 01, best_model_dir contents: ['best_model_run00.pt', 'best_model_run01.pt']
  [DEBUG] After save_ckp for run 01, best_model_dir contents: ['best_model_run00.pt', 'best_model_run01.pt']
  Run 01 | Early stop @ ep 330 | best Val AUC 0.7837
Run 02 | Task: SR-ARE       | Test AUC: 0.7345 | Val AUC: 0.7837 | Run Time: 4.73m | Total Time: 8.82m
  [DEBUG] After save_ckp for run 02, best_model_dir contents: ['

KeyboardInterrupt: 

#Final Experiment Summary



In [ ]:
# total_exp_time = (time.time() - experiment_start) / 60

print(f'\n{"="*60}')
print(f'EXPERIMENT COMPLETE — Model: {MOD_NAME} | Task: {TASK_NAME}')
print(f'{"="*60}')
print(f'  Mean Test AUC : {np.mean(all_run_aucs):.4f} ± {np.std(all_run_aucs):.4f}')
print(f'  Min Test AUC  : {np.min(all_run_aucs):.4f}')
print(f'  Max Test AUC  : {np.max(all_run_aucs):.4f}')
print(f'------------------------------------------------------------')
# print(f'  Mean Time/Run : {np.mean(all_run_times)/60:.2f} m')
# print(f'  Total Duration: {total_exp_time:.2f} m')
print(f'  Saved Results : {RESULTS_DIR}/{TASK_NAME}_runs_summary.csv')
print(f'{"="*60}')

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/Research/HybridGNN/TOX/SR-ARE/results_tox21/SR-ARE/sage_gin_gin/"

In [ ]:
import os
import pandas as pd
from tensorboard.backend.event_processing import event_accumulator

def load_tensorboard_data(log_dir):
    """
    Parses TensorBoard event files in a directory and returns a DataFrame.
    """
    ea = event_accumulator.EventAccumulator(
        log_dir,
        size_guidance={event_accumulator.SCALARS: 0} # 0 loads all scalar points
    )
    ea.Reload()

    # Get available scalar tags (e.g., 'Loss/train', 'AUC/val', 'Learning Rate')
    tags = ea.Tags()['scalars']

    data = []
    for tag in tags:
        for event in ea.Scalars(tag):
            data.append({
                'metric': tag,
                'step': event.step,
                'value': event.value,
                'wall_time': event.wall_time
            })

    return pd.DataFrame(data)

# Example: Read run_00 logs
log_path = os.path.join(RESULTS_DIR, TASK_NAME, MOD_NAME, 'run_00')
df_events = load_tensorboard_data(log_path)

# Pivot to display metrics side-by-side per step
df_pivot = df_events.pivot(index='step', columns='metric', values='value')
print(df_pivot.head())

In [ ]:
import os
csv_file_path = os.path.join(RESULTS_DIR, f'{TASK_NAME}_runs_summary.csv')

if not os.path.exists(csv_file_path):
    print(f"No summary file found for task {TASK_NAME} in {RESULTS_DIR}. Run the experiment first!")
else:
    df_res = pd.read_csv(csv_file_path)

    df_res['test_auc'] = df_res['test_auc'].astype(float)
    df_res['best_val_auc'] = df_res['best_val_auc'].astype(float)

    # 1.Summary Table (can be used for multi-class)
    print("==================================================================")
    print(f"            PERFORMANCE SUMMARY FOR TASK: {TASK_NAME.upper()}            ")
    print("==================================================================")
    summary_table = df_res.groupby('task')['test_auc'].agg(
        Mean='mean', Std='std', Median='median', Min='min', Max='max', Runs='count'
    ).reset_index()

    print(summary_table.to_string(index=False))

    overall_mean = df_res['test_auc'].mean()
    overall_std = df_res['test_auc'].std()
    print("-" * 66)
    print(f"AVERAGE TEST AUC FOR TASK {TASK_NAME}: {overall_mean:.4f} \u00b1 {overall_std:.4f}\n")

    # Visualizations Dashboard
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    sns.set_theme(style="whitegrid")

    # Plot A: Boxplot of Test AUC by Task
    sns.boxplot(data=df_res, x='task', y='test_auc', ax=axes[0, 0], palette="Set2")
    axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45, ha='right')
    axes[0, 0].set_title(f"Test ROC-AUC Distribution for Task {TASK_NAME}", fontsize=14, fontweight='bold') # Changed title
    axes[0, 0].set_ylabel("ROC-AUC")

    # Plot B: Mean Performance Comparison (Ranked)
    ranked_summary = summary_table.sort_values(by='Mean', ascending=False)
    sns.barplot(data=ranked_summary, x='Mean', y='task', ax=axes[0, 1], palette="Blues_r")
    axes[0, 1].set_title(f"Ranked Average Test AUC for Task {TASK_NAME}", fontsize=14, fontweight='bold') # Changed title
    axes[0, 1].set_xlabel("Mean ROC-AUC")

    # Plot C: Validation Trajectories
    for idx, row in df_res.iterrows():
        try:
            val_hist = json.loads(row['val_auc_history'])
            axes[1, 0].plot(range(len(val_hist)), val_hist, alpha=0.15, color='gray')
        except:
            pass

    all_val_hists = [json.loads(h) for h in df_res['val_auc_history'].dropna()]
    max_len = max([len(h) for h in all_val_hists])
    padded_hists = [h + [np.nan]*(max_len - len(h)) for h in all_val_hists]
    mean_val_curve = np.nanmean(padded_hists, axis=0)

    axes[1, 0].plot(range(len(mean_val_curve)), mean_val_curve, color='red', linewidth=2.5, label='Mean Validation AUC')
    axes[1, 0].set_title(f"Validation AUC Trajectories for Task {TASK_NAME}", fontsize=14, fontweight='bold') # Changed title
    axes[1, 0].set_xlabel("Evaluation Step")
    axes[1, 0].set_ylabel("Validation ROC-AUC")
    axes[1, 0].legend()

    # Plot D: Training Loss Trajectories
    for idx, row in df_res.iterrows():
        try:
            loss_hist = json.loads(row['train_loss_history'])
            axes[1, 1].plot(range(len(loss_hist)), loss_hist, alpha=0.15, color='steelblue')
        except:
            pass

    axes[1, 1].set_title(f"Training Loss Trajectories for Task {TASK_NAME}", fontsize=14, fontweight='bold') # Changed title
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("BCE Loss")

    plt.tight_layout()
    plot_path = os.path.join(RESULTS_DIR, f'per_task_evaluation_dashboard_{TASK_NAME}.pdf')
    plt.savefig(plot_path, format ='pdf', bbox_inches='tight')
    plt.show()

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def load_and_analyze_task_results(csv_path):
    """
    Loads run summaries, unpacks JSON history arrays, and prints statistics.
    """
    df = pd.read_csv(csv_path)

    # Safely unpack JSON string columns back into Python lists
    df['train_loss_history'] = df['train_loss_history'].apply(json.loads)
    df['val_auc_history'] = df['val_auc_history'].apply(json.loads)

    # 1. Metric Summaries Across Runs
    print(f"=== Analysis for {df['task'].iloc[0]} ({df['model'].iloc[0]}) ===")
    print(f"Total Runs Completed: {len(df)}")
    print(f"Mean Test AUC:     {df['test_auc'].mean():.4f} +/- {df['test_auc'].std():.4f}")
    print(f"Mean Best Val AUC: {df['best_val_auc'].mean():.4f} +/- {df['best_val_auc'].std():.4f}")

    # 2. Plot Aggregated Learning Curves Across All Runs
    plt.figure(figsize=(12, 5))

    # Plot Training Loss
    plt.subplot(1, 2, 1)
    for idx, row in df.iterrows():
        plt.plot(row['train_loss_history'], alpha=0.3, label=f"Run {row['run']}" if len(df) <= 5 else "")

    # Mean loss curve across runs
    max_epochs = max(len(h) for h in df['train_loss_history'])
    padded_losses = np.array([h + [np.nan] * (max_epochs - len(h)) for h in df['train_loss_history']])
    mean_loss = np.nanmean(padded_losses, axis=0)
    plt.plot(mean_loss, color='black', linewidth=2, label='Mean Loss')
    plt.title('Train Loss History')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()

    # Plot Validation AUC
    plt.subplot(1, 2, 2)
    for idx, row in df.iterrows():
        plt.plot(row['val_auc_history'], alpha=0.3)

    padded_aucs = np.array([h + [np.nan] * (max_epochs - len(h)) for h in df['val_auc_history']])
    mean_auc = np.nanmean(padded_aucs, axis=0)
    plt.plot(mean_auc, color='black', linewidth=2, label='Mean Val AUC')
    plt.title('Validation AUC History')
    plt.xlabel('Eval Step')
    plt.ylabel('AUC')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()

    plt.tight_layout()
    plt.show()

# Example Usage:
load_and_analyze_task_results(csv_file_path)

## 11 · Statistical significance

> Add blockquote


Run after you have 30-run results for all 4 baselines. Paste the AUC lists below.

In [ ]:
import scipy.stats as stats

# Ensure RESULTS_DIR and NAME match your notebook variables
# Changed to read the task-specific summary file, as generated by the current experiment.
csv_path = os.path.join(RESULTS_DIR, f'{TASK_NAME}_runs_summary.csv')

if os.path.exists(csv_path):
    try:
        df_results = pd.read_csv(csv_path)

        # Check if 'test_auc' column exists, otherwise inform the user.
        # 'mod_auc' is not generated by the current experiment.
        if 'test_auc' not in df_results.columns:
            print(f"Error: 'test_auc' column not found in {csv_path}. Expected for analysis. Please check the CSV structure.")
        elif 'model' not in df_results.columns:
            print(f"Error: 'model' column not found in {csv_path}. Expected for analysis.")
        else:
            # Extract hybrid model scores using 'test_auc' instead of 'mod_auc'
            hybrid_scores = df_results[df_results['model'] == MOD_NAME]['test_auc'].values

            if len(hybrid_scores) == 0:
                print(f"No results found for model '{MOD_NAME}' in {csv_path}.")
            elif len(hybrid_scores) < 2:
                print(f"Not enough runs for model '{MOD_NAME}' ({len(hybrid_scores)} runs) to calculate meaningful statistics (need at least 2).")
            else:
                print(f'Hybrid ({MOD_NAME}): median={np.median(hybrid_scores):.4f} ± {np.std(hybrid_scores):.4f}\n')

                header = f'{"Model":<12}  {"Median":>7}  {"Std":>7}  {"p-value":>12}  {"Significant?":>13}'
                print(header)
                print('-' * len(header))

                models_in_file = df_results['model'].unique()
                compared_any = False
                for m in models_in_file:
                    if m == MOD_NAME:
                        continue

                    # Use 'test_auc' instead of 'mod_auc'
                    baucs = df_results[df_results['model'] == m]['test_auc'].values
                    if len(baucs) < 2:
                        print(f"Skipping comparison for model '{m}': Less than 2 valid 'test_auc' scores.")
                        continue

                    # Perform Mann-Whitney U test (non-parametric, suitable for non-normal distributions)
                    # 'alternative='greater'' tests if hybrid_scores are stochastically greater than baucs
                    _, p = stats.mannwhitneyu(hybrid_scores, baucs, alternative='greater')
                    sig = 'YES p<0.05' if p < 0.05 else 'no'
                    print(f'{m:<12}  {np.median(baucs):>7.4f}  {np.std(baucs):>7.4f}  {p:>12.2e}  {sig:>13}')
                    compared_any = True

                if not compared_any and len(models_in_file) <= 1:
                    print(f"No other models found in {csv_path} for comparison with '{MOD_NAME}'.")
                    print("To perform statistical comparisons, ensure this file contains results for multiple models.")
    except pd.errors.ParserError as e:
        print(f"Error parsing CSV file {csv_path}: {e}")
        print("Please check the format of the CSV file. It might be malformed.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
else:
    print(f"Summary CSV not found at {csv_path}. Please ensure the experiment (Cell 10) has been run and generated '{TASK_NAME}_runs_summary.csv'.")

## 12 · Results plots

In [ ]:
# MOD_NAME = 'sage_gin_gin'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5)) # Changed to create 2 subplots

# --- Test AUC Distribution across N Runs ---
# summary_df and filter for the current model NAME
summary_df  = pd.read_csv(os.path.join(RESULTS_DIR, f'{TASK_NAME}_runs_summary.csv'))
hybrid_rows = summary_df[summary_df['model'] == MOD_NAME]

# all_run_aucs from the loaded summary_df for robustness
all_run_aucs = hybrid_rows['test_auc'].values

sns.boxplot(y=all_run_aucs, ax=axes[0], color='steelblue', width=0.35, boxprops=dict(alpha=0.7))
sns.stripplot(y=all_run_aucs, ax=axes[0], color='navy', alpha=0.8, jitter=0.1, size=6)

# Summary statistics reference line
median_auc = np.median(all_run_aucs)
std_auc = np.std(all_run_aucs)
axes[0].axhline(median_auc, color='red', ls='--',
                label=f'Median: {median_auc:.4f} \u00b0.0179 {std_auc:.4f}')

axes[0].set_title(f'Test AUC Spread ({N_RUNS} Runs)\n{TASK_NAME} | {MOD_NAME}')
axes[0].set_ylabel(f'Test ROC-AUC ({TASK_NAME})')
axes[0].legend(loc='lower right')

# ---  Validation Trajectories over Training ---
#  val_auc_history from JSON strings
all_val_hists = [json.loads(h) for h in hybrid_rows['val_auc_history'].dropna()]

if all_val_hists:
    max_len = max([len(h) for h in all_val_hists])
    padded_hists = [h + [np.nan]*(max_len - len(h)) for h in all_val_hists]

    mean_trajectory = np.nanmean(padded_hists, axis=0)
    std_trajectory = np.nanstd(padded_hists, axis=0)

    epochs = [(i + 1) * EVAL_EVERY for i in range(max_len)]

    # individual run trajectories
    for hist in all_val_hists:
        run_epochs = [(i + 1) * EVAL_EVERY for i in range(len(hist))]
        axes[1].plot(run_epochs, hist, alpha=0.3, color='steelblue', lw=1.2)

    # overall mean trajectory across runs with std dev band
    axes[1].plot(epochs, mean_trajectory, color='navy', lw=2.5, label='Mean Val AUC')
    axes[1].fill_between(epochs, mean_trajectory - std_trajectory, mean_trajectory + std_trajectory,
                         color='navy', alpha=0.1, label='\u00b11 Std Dev')

    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel(f'Val ROC-AUC ({TASK_NAME})')
    axes[1].set_title(f'Val AUC Trajectory ({N_RUNS} Runs)\n{TASK_NAME}')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "No validation AUC history found for plotting",
                 horizontalalignment='center', verticalalignment='center',
                 transform=axes[1].transAxes, fontsize=12, color='gray')
    axes[1].set_title(f'Val AUC Trajectory ({N_RUNS} Runs)\n{TASK_NAME}')

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, f'{TASK_NAME}_{MOD_NAME}_results.pdf')
# plt.savefig(plot_path, format='pdf', bbox_inches='tight')
plt.show()
print(f'Saved plot to {plot_path}')

In [ ]:
summary_df.info()

In [ ]:
summary_df.head()

In [ ]:
plt.figure(figsize=(7, 4.5))
mean_curve = hybrid_rows[auc_cols].mean(axis=0).values
std_curve  = hybrid_rows[auc_cols].std(axis=0).values

plt.plot(epochs, mean_curve, color='steelblue', label='Mean Val AUC', lw=2)
plt.fill_between(epochs, mean_curve - std_curve, mean_curve + std_curve,
                 color='steelblue', alpha=0.2, label='±1 Std Dev')
plt.axhline(max(mean_curve), color='red', linestyle=':', label=f'Peak Mean Val AUC: {max(mean_curve):.4f}')

plt.xlabel('Epochs')
plt.ylabel(f'Validation AUC ({TASK_NAME})')
plt.title(f'Learning Curve Stability ({N_RUNS} Runs) — {TASK_NAME}')
plt.legend()
plt.savefig(os.path.join(RESULTS_DIR, f'{TASK_NAME}_val_band.pdf'), bbox_inches='tight')
plt.show()

## 13 · Load a saved checkpoint (optional)
Use this cell to reload the best model from any run — e.g. to run inference or inspect weights.
The checkpoint saved in Cell 10 contains the full model state including which layers were used.


In [ ]:
# best_layer_types = ['sage', 'gin', 'gin']
# best_hidden = 520
# best_dropout = np.round(0.43479245061434324, 4)
# best_lr =np.round(0.0003585966617440445, 5)

In [ ]:
# MOD_NAME = 'sage_gin_gin'

In [ ]:
torch.serialization.add_safe_globals([np._core.multiarray.scalar])

# ── Setup paths ──────────────
load_name = MOD_NAME
base_checkpoint_dir = os.path.join(CHECKPOINT_DIR, TASK_NAME)
# all checkpoints are saved
task_ckpt_dir = os.path.join(base_checkpoint_dir, load_name)
# for saving the best model
task_model_dir = os.path.join(base_checkpoint_dir, load_name + '_best')

print(f"Inspecting directories:")
if os.path.exists(task_ckpt_dir):
    print(f"  Contents of {task_ckpt_dir}: {os.listdir(task_ckpt_dir)}")
else:
    print(f"  Error: Checkpoint directory not found: {task_ckpt_dir}")

if os.path.exists(task_model_dir):
    print(f"  Contents of {task_model_dir}: {os.listdir(task_model_dir)}")
else:
    print(f"  Error: Best model directory not found: {task_model_dir}")


# ── Attempt to load the best model from any run ──────────────────────────────
loaded_model = None
loaded_opt = None
start_epoch = None

for run_id_to_try in range(N_RUNS):
    # First try to load from the 'best' directory
    ckpt_path_best = os.path.join(task_model_dir, f'best_model_run{run_id_to_try:02d}.pt')
    # If not found, try to load from the general checkpoint directory
    ckpt_path_general = os.path.join(task_ckpt_dir, f'model_run{run_id_to_try:02d}.pt')

    if os.path.exists(ckpt_path_best):
        print(f"\nAttempting to load best model from: {ckpt_path_best}")
        ckpt_path = ckpt_path_best
        break
    elif os.path.exists(ckpt_path_general):
        print(f"\nAttempting to load general model from: {ckpt_path_general}")
        ckpt_path = ckpt_path_general
        break
else:
    print(f"\nError: No checkpoint files found for model '{load_name}' in either '{task_model_dir}' or '{task_ckpt_dir}' for any of the {N_RUNS} runs.")
    print("Please ensure runs completed successfully and saved their models.")
    exit() # Exit if no checkpoint is found

# If a checkpoint path was found and assigned:
if 'ckpt_path' in locals():
    # Reconstruct the model with the same architecture
    loaded_model = HybridGNN(best_layer_types, best_hidden, best_dropout).to(DEVICE)
    loaded_opt   = torch.optim.Adam(loaded_model.parameters(), lr=best_lr)

    loaded_model, loaded_opt, start_epoch = load_ckp(ckpt_path, loaded_model, loaded_opt)

    # Verify
    auc = evaluate_auc(loaded_model, test_loader, DEVICE, task_cols)
    print(f'Loaded run {run_id_to_try} checkpoint (trained to epoch {start_epoch})')
    print(f'Test AUC on reload: {auc:.4f}')


In [ ]:
print(task_ckpt_dir)
print(task_model_dir)
print(ckpt_path)